# Day 1 — Exploratory Data Analysis

This notebook is the first look at the well-log data after the loader and
joiner have done their job. It is the place to develop intuition before
writing analytical code in `src/`.

**Goal of this notebook:** confirm what the data looks like, where the
missingness is, how `perm` is distributed, and what the zones look like
across wells. Findings get promoted into the deliverable report later.

In [ ]:
import sys
from pathlib import Path

# Allow importing from src/ when running the notebook from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_all_wells, load_zones
from src.data.joiner import build_master_table

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load everything
RAW = ROOT / 'data' / 'raw'
logs = load_all_wells(RAW)
zones = load_zones(RAW)
master = build_master_table(logs, zones)
master.head()

## 1. Inventory

How big is each well, and at what depth range?

In [ ]:
(master
  .groupby('well')
  .agg(n_samples=('depth', 'count'),
       depth_min=('depth', 'min'),
       depth_max=('depth', 'max'),
       zones=('zone', lambda s: sorted(s.dropna().unique().tolist())))
 )

## 2. Missingness pattern

In [ ]:
missing_pct = (
    master.groupby('well')[['vsh', 'phit', 'sw', 'perm']]
          .apply(lambda g: g.isna().mean() * 100)
)
missing_pct.style.format('{:.2f}%').background_gradient(cmap='Reds')

## 3. Distributions of the four log curves

Note: `perm` is plotted on a log scale because it spans 5+ orders of
magnitude. This is why downstream clustering uses `log10(perm)`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ['vsh', 'phit', 'sw', 'perm']):
    if col == 'perm':
        ax.hist(np.log10(master[col].dropna().clip(lower=1e-3)), bins=60, color='steelblue', edgecolor='white')
        ax.set_xlabel('log10(perm) [mD]')
    else:
        ax.hist(master[col].dropna(), bins=60, color='steelblue', edgecolor='white')
        ax.set_xlabel(col)
    ax.set_title(col)
    ax.set_ylabel('count')
plt.tight_layout()

## 4. Per-zone summary

First-cut comparison of the zones. Which zones tend to have clean,
porous, permeable rock?

In [ ]:
zone_summary = (
    master.dropna(subset=['zone'])
          .groupby('zone')
          .agg(n_samples=('depth', 'count'),
               mean_vsh=('vsh', 'mean'),
               mean_phit=('phit', 'mean'),
               mean_sw=('sw', 'mean'),
               median_perm=('perm', 'median'))
)
zone_summary

## 5. Cross-curve relationships

We expect: higher phit ↔ lower vsh ↔ higher perm. If this doesn't hold,
the data may have a measurement issue.

In [ ]:
subset = master.dropna(subset=['vsh', 'phit', 'perm']).sample(min(3000, len(master)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(subset.vsh, subset.phit, s=3, alpha=0.3)
axes[0].set_xlabel('vsh'); axes[0].set_ylabel('phit'); axes[0].set_title('vsh vs phit')
axes[1].scatter(subset.phit, np.log10(subset.perm.clip(lower=1e-3)), s=3, alpha=0.3)
axes[1].set_xlabel('phit'); axes[1].set_ylabel('log10(perm)'); axes[1].set_title('phit vs log perm')
plt.tight_layout()

## Notes / TODO

- [ ] Check whether `perm == 15000` is a sensor cap (already flagged in
      `data/quality.py`).
- [ ] Decide on missing-value policy for analytics (drop vs interpolate vs
      flag-and-keep). Per-column decision is fine.
- [ ] Pick the most-populated zone for Part D sub-zone clustering.